# FIID acquirer group A/B check (`RSHB`)

Notebook shows the difference between:
- all operations on selected merchant terminals;
- only operations where `c_fiid_acq_grp = 'RSHB'`.

Purpose: quantify potential `"foreign"` turnover if filter is not applied.

In [ ]:
import re
import pandas as pd
from decimal import Decimal, InvalidOperation
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
report_month = '2026-02-01'
month_start = pd.to_datetime(report_month).strftime('%Y-%m-%d')
month_end = (pd.to_datetime(report_month) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

sa_type = 'SA'
mem_limit = '10g'

# Auto-pick INN from Excel by top turnover.
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_col_inn = 'ИНН'
excel_col_trx_sum = 'Сумма операций'
use_manual_target_inn = False
manual_target_inn = '7708044880'

print('report_month =', report_month)
print('month_start =', month_start)
print('month_end =', month_end)
print('sa_type =', sa_type)
print('mem_limit =', mem_limit)
print('excel_path =', excel_path)
print('use_manual_target_inn =', use_manual_target_inn)
print('manual_target_inn =', manual_target_inn)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_base24_fiids')

print('Impala initialized')

## 1) Pick target INN from Excel (top turnover)

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

if use_manual_target_inn:
    target_inn = normalize_numeric_str(manual_target_inn)
    if target_inn is None:
        raise ValueError('manual_target_inn is invalid')
    pick_inn_df = pd.DataFrame([{'inn': target_inn, 'excel_turnover': None, 'rows_cnt': None, 'source': 'manual'}])
else:
    excel_raw = pd.read_excel(excel_path)
    excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

    req_cols = [excel_col_inn, excel_col_trx_sum]
    missing = [c for c in req_cols if c not in excel_raw.columns]
    if missing:
        raise ValueError(f'Missing required Excel columns: {missing}')

    top_df = excel_raw[[excel_col_inn, excel_col_trx_sum]].copy()
    top_df.columns = ['inn_raw', 'trx_sum_raw']
    top_df['inn'] = top_df['inn_raw'].apply(normalize_numeric_str)
    top_df['trx_sum'] = pd.to_numeric(top_df['trx_sum_raw'], errors='coerce').fillna(0)

    top_df = top_df[(top_df['inn'].notna()) & (top_df['inn'] != '')].copy()
    top_agg = top_df.groupby('inn', as_index=False).agg(
        excel_turnover=('trx_sum', 'sum'),
        rows_cnt=('trx_sum', 'count')
    ).sort_values(['excel_turnover', 'rows_cnt', 'inn'], ascending=[False, False, True])

    if len(top_agg) == 0:
        raise ValueError('No valid INN in Excel for auto-pick')

    target_inn = str(top_agg.iloc[0]['inn'])
    pick_inn_df = top_agg.head(20).copy()

print('target_inn =', target_inn)
print('Top INN by Excel turnover:')
display(pick_inn_df.head(20))

## 2) Build base perimeter (client SA contracts + terminal ids)

In [ ]:
sql_base = f"""
with cmp as (
  select
    cast(c.n_cmp as string) as n_cmp_client,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from ods_alpha.scd1_companies c
  where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{target_inn}'
),
agr as (
  select
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.c_agr_number as string) as contract_number
  from ods_alpha.scd1_agreements a
  join cmp c on c.n_cmp_client = cast(a.n_cmp_client as string)
  where upper(trim(cast(a.acq_class as string))) = '{sa_type}'
    and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
),
retl as (
  select
    cast(m.n_cmp as string) as n_cmp_client,
    cast(m.c_nmrc as string) as retl_id
  from ods_alpha.scd1_merchants m
  join cmp c on c.n_cmp_client = cast(m.n_cmp as string)
  where m.c_nmrc is not null
),
terms as (
  select distinct
    cast(t.c_nter as string) as term_id,
    cast(t.c_nmrc as string) as retl_id
  from ods_alpha.scd1_trx t
  where t.c_nter is not null
)
select
  c.inn,
  a.n_cmp_client,
  a.n_agr,
  a.contract_number,
  r.retl_id,
  tm.term_id
from cmp c
left join agr a on a.n_cmp_client = c.n_cmp_client
left join retl r on r.n_cmp_client = c.n_cmp_client
left join terms tm on tm.retl_id = r.retl_id
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    base_df = imp.fetch(sql_base)

display(base_df.head(30))
print('base_rows =', len(base_df))
print('distinct_n_agr =', int(base_df['n_agr'].dropna().nunique()) if len(base_df) else 0)
print('distinct_retl_id =', int(base_df['retl_id'].dropna().nunique()) if len(base_df) else 0)
print('distinct_term_id =', int(base_df['term_id'].dropna().nunique()) if len(base_df) else 0)

## 3) A/B metrics: all terminal operations vs `c_fiid_acq_grp='RSHB'

In [ ]:
if len(base_df) == 0:
    ab_summary = pd.DataFrame()
    fiid_breakdown = pd.DataFrame()
else:
    term_ids = sorted(base_df['term_id'].dropna().astype(str).unique().tolist())

    if len(term_ids) == 0:
        ab_summary = pd.DataFrame()
        fiid_breakdown = pd.DataFrame()
    else:
        term_chunks = [term_ids[i:i+1200] for i in range(0, len(term_ids), 1200)]
        parts = []

        for chunk in term_chunks:
            in_list = ', '.join(["'" + t.replace("'", "''") + "'" for t in chunk])
            sql_ab = f"""
            with trx_base as (
              select
                cast(t.n_trx as string) as n_trx,
                cast(t.c_nter as string) as term_id,
                cast(t.c_nmrc as string) as retl_id,
                cast(t.n_amt_src as double) as n_amt_src,
                cast(t.c_fiid_acq as string) as c_fiid_acq,
                cast(t.c_fiid_iss as string) as c_fiid_iss,
                cast(t.c_trx_type as string) as c_trx_type,
                cast(t.c_trx_class as string) as c_trx_class,
                cast(t.cf_trx_stat as string) as cf_trx_stat,
                cast(t.ods_deleted_flg as string) as ods_deleted_flg,
                to_date(cast(t.d_trx_orig as timestamp)) as trx_dt
              from ods_alpha.scd1_trx t
              where cast(t.c_nter as string) in ({in_list})
                and to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
                and t.c_nter is not null
                and t.c_trx_class = 'SA'
                and t.c_trx_type = 'S01'
                and coalesce(t.cf_trx_stat, '') <> 'R'
                and coalesce(t.ods_deleted_flg, '0') <> '1'
            ),
            fiid as (
              select
                cast(f.c_fiid as string) as c_fiid,
                cast(f.c_fiid_grp as string) as c_fiid_grp
              from ods_alpha.scd1_base24_fiids f
            ),
            x as (
              select
                b.*,
                coalesce(fa.c_fiid_grp, 'UNKNOWN') as c_fiid_acq_grp,
                coalesce(fi.c_fiid_grp, 'UNKNOWN') as c_fiid_iss_grp
              from trx_base b
              left join fiid fa on fa.c_fiid = b.c_fiid_acq
              left join fiid fi on fi.c_fiid = b.c_fiid_iss
            )
            select
              c_fiid_acq_grp,
              count(distinct n_trx) as trx_cnt,
              sum(coalesce(n_amt_src, 0.0)) as trx_sum
            from x
            group by c_fiid_acq_grp
            """
            with imp:
                imp.execute(f"set MEM_LIMIT={mem_limit}")
                parts.append(imp.fetch(sql_ab))

        fiid_breakdown = pd.concat(parts, ignore_index=True) if len(parts) else pd.DataFrame(columns=['c_fiid_acq_grp','trx_cnt','trx_sum'])
        fiid_breakdown = fiid_breakdown.groupby('c_fiid_acq_grp', as_index=False).agg({'trx_cnt':'sum', 'trx_sum':'sum'})
        fiid_breakdown = fiid_breakdown.sort_values('trx_sum', ascending=False)

        total_trx_cnt = int(fiid_breakdown['trx_cnt'].sum()) if len(fiid_breakdown) else 0
        total_trx_sum = float(fiid_breakdown['trx_sum'].sum()) if len(fiid_breakdown) else 0.0

        rshb_row = fiid_breakdown[fiid_breakdown['c_fiid_acq_grp'] == 'RSHB']
        rshb_trx_cnt = int(rshb_row['trx_cnt'].sum()) if len(rshb_row) else 0
        rshb_trx_sum = float(rshb_row['trx_sum'].sum()) if len(rshb_row) else 0.0

        foreign_trx_cnt = total_trx_cnt - rshb_trx_cnt
        foreign_trx_sum = total_trx_sum - rshb_trx_sum

        ab_summary = pd.DataFrame([
            {'metric': 'all_ops_trx_cnt', 'value': total_trx_cnt},
            {'metric': 'all_ops_trx_sum', 'value': total_trx_sum},
            {'metric': 'rshb_only_trx_cnt', 'value': rshb_trx_cnt},
            {'metric': 'rshb_only_trx_sum', 'value': rshb_trx_sum},
            {'metric': 'foreign_trx_cnt', 'value': foreign_trx_cnt},
            {'metric': 'foreign_trx_sum', 'value': foreign_trx_sum},
            {'metric': 'foreign_trx_cnt_share_pct', 'value': (foreign_trx_cnt / total_trx_cnt * 100.0) if total_trx_cnt else 0.0},
            {'metric': 'foreign_trx_sum_share_pct', 'value': (foreign_trx_sum / total_trx_sum * 100.0) if total_trx_sum else 0.0},
        ])

display(ab_summary)
print('FIID acquirer group breakdown:')
display(fiid_breakdown)

## 4) Detail sample: operations not in `RSHB` acquirer group

In [ ]:
if len(base_df) == 0:
    foreign_sample_df = pd.DataFrame()
else:
    term_ids = sorted(base_df['term_id'].dropna().astype(str).unique().tolist())

    if len(term_ids) == 0:
        foreign_sample_df = pd.DataFrame()
    else:
        sample_terms = term_ids[:1000]
        in_list = ', '.join(["'" + t.replace("'", "''") + "'" for t in sample_terms])

        sql_foreign_sample = f"""
        with trx_base as (
          select
            cast(t.n_trx as string) as n_trx,
            cast(t.c_nter as string) as term_id,
            cast(t.c_nmrc as string) as retl_id,
            cast(t.n_amt_src as double) as n_amt_src,
            cast(t.c_fiid_acq as string) as c_fiid_acq,
            cast(t.c_fiid_iss as string) as c_fiid_iss,
            cast(t.c_trx_type as string) as c_trx_type,
            cast(t.c_trx_class as string) as c_trx_class,
            cast(t.cf_trx_stat as string) as cf_trx_stat,
            cast(t.ods_deleted_flg as string) as ods_deleted_flg,
            to_date(cast(t.d_trx_orig as timestamp)) as trx_dt
          from ods_alpha.scd1_trx t
          where cast(t.c_nter as string) in ({in_list})
            and to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
            and t.c_nter is not null
            and t.c_trx_class = 'SA'
            and t.c_trx_type = 'S01'
            and coalesce(t.cf_trx_stat, '') <> 'R'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
        )
        select
          b.n_trx,
          b.trx_dt,
          b.term_id,
          b.retl_id,
          b.n_amt_src,
          b.c_fiid_acq,
          coalesce(fa.c_fiid_grp, 'UNKNOWN') as c_fiid_acq_grp,
          b.c_fiid_iss,
          coalesce(fi.c_fiid_grp, 'UNKNOWN') as c_fiid_iss_grp
        from trx_base b
        left join ods_alpha.scd1_base24_fiids fa on cast(fa.c_fiid as string) = b.c_fiid_acq
        left join ods_alpha.scd1_base24_fiids fi on cast(fi.c_fiid as string) = b.c_fiid_iss
        where coalesce(fa.c_fiid_grp, 'UNKNOWN') <> 'RSHB'
        order by b.trx_dt desc, b.n_trx
        limit 500
        """

        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            foreign_sample_df = imp.fetch(sql_foreign_sample)

print('foreign_sample_rows =', len(foreign_sample_df))
display(foreign_sample_df)